> **ℹ️ Note**
>
**Durée estimée** : 3 heures  
**Prérequis** : Notions 7.1 à 7.5 + Module 6 (Deep Learning)  
**Objectif** : comprendre LoRA, savoir construire un dataset, et surtout savoir **quand** fine-tuner (et quand NE PAS le faire)


## 📋 Ce que tu sauras faire à la fin

- Savoir **quand fine-tuner** (et quand ne pas le faire)
- Comprendre l'intuition de **LoRA** (Low-Rank Adaptation)
- Utiliser **PEFT** (Parameter-Efficient Fine-Tuning) en pratique
- **Préparer un dataset** de fine-tuning (format JSONL, train/val)
- **Évaluer** un modèle fine-tuné proprement
- Connaître les approches modernes : **RLHF, DPO, Constitutional AI**

## 🎯 Pourquoi cette notion ?

Le fine-tuning est la **technique la plus mal utilisée** de la boîte à outils LLM. 80% des équipes qui se lancent dans un fine-tuning auraient **mieux fait** d'utiliser du prompt engineering ou du RAG.

Ce que tu vas apprendre ici :

- Savoir **reconnaître** les cas où le fine-tuning vaut le coup
- Économiser du **temps et de l'argent** en choisissant la bonne technique
- Quand fine-tuner : le faire **efficacement** avec LoRA
- **Éviter les pièges** classiques

## 🛠️ Mise en route

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import tempfile
from pathlib import Path

# Dossier temporaire portable (Windows / macOS / Linux / Colab)
TMP = Path(tempfile.gettempdir())

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

print("✅ Environnement prêt")

> **🎯 Important**
>
## 📦 Librairies nécessaires

```bash
pip install transformers peft datasets accelerate bitsandbytes
```

**Note** : un entraînement réel nécessite un GPU (Colab T4 gratuit suffit pour LoRA sur petit modèle).


# 1. Quand fine-tuner ? (et quand NE PAS le faire)

## 🤔 Les 3 approches pour "spécialiser" un LLM

Quand tu veux qu'un LLM soit bon sur **ton cas d'usage**, tu as **3 options**, par ordre de coût croissant :

| Approche | Coût | Temps | Quand l'utiliser |
|---|:---:|:---:|---|
| **Prompt engineering** | 💰 | ⚡ minutes | Toujours **commencer** par ça |
| **RAG** | 💰💰 | 🕐 jours | Besoin d'accès à des données privées |
| **Fine-tuning** | 💰💰💰💰 | 📆 semaines | Comportement ou style spécifique |

## ❌ Les 4 cas où il NE FAUT PAS fine-tuner

### Cas 1 : "Le LLM ne connaît pas mes données"

**❌ Mauvaise idée** : fine-tuner sur toute ta doc interne.

**✅ Bonne approche** : **RAG** (Notion 7.4). C'est **beaucoup** plus efficace et maintenable.

**Pourquoi** : le fine-tuning pour injecter des connaissances est **inefficace** (forget catastrophique), **coûteux** (besoin de refaire à chaque MAJ de doc) et **risqué** (hallucinations).

### Cas 2 : "Le LLM fait des erreurs sur des cas particuliers"

**❌ Mauvaise idée** : fine-tuner pour "corriger".

**✅ Bonne approche** : améliorer le **prompt** (few-shot, contraintes explicites).

### Cas 3 : "Je veux un chatbot avec la personnalité de ma marque"

**❌ Mauvaise idée** : fine-tuner pour "ajuster le ton".

**✅ Bonne approche** : un bon **system prompt** suffit.

### Cas 4 : "Je veux améliorer les réponses"

**❌ Trop vague** pour justifier un fine-tuning.

**✅ Approche** : **identifier précisément** ce qui ne va pas, tester prompt engineering d'abord.

## ✅ Les 4 vrais cas où fine-tuner vaut le coup

### Cas 1 : Format de sortie très spécifique

Tu veux que le LLM sorte **toujours** du JSON avec un schéma complexe, ou du SQL très particulier à ton dialecte.

**Exemple** : fine-tuner un modèle pour générer des requêtes Elasticsearch à partir de langage naturel.

### Cas 2 : Domaine ultra-spécialisé

Ton domaine a un **vocabulaire** et des **conventions** non couverts par les modèles généralistes :
- Médical (radiologie, anatomopathologie)
- Juridique (textes de loi, jurisprudence)
- Scientifique (chimie, biologie moléculaire)

### Cas 3 : Coût par requête à optimiser

Tu as un **petit modèle** (Llama 3.2 1B) fine-tuné qui **bat** un gros modèle generaliste sur ta tâche. 100× moins cher en inférence.

**Exemple** : un classifier de tickets qu'on appelle 1 million de fois/jour.

### Cas 4 : Style ou comportement systématique

Tu veux une **voix de marque** très marquée (ex: Duolingo's chatty tone), ou un **comportement** spécifique (ex: toujours poser 2 questions avant de répondre).

## 🧠 La règle d'or

> **Commence toujours par prompt engineering + RAG.**  
> Ne fine-tune QUE si tu as mesuré que ces techniques sont insuffisantes.

90% des projets réussis utilisent un **gros LLM + bon prompt + RAG**, sans jamais fine-tuner.

# 2. Full fine-tuning vs LoRA

## 🐘 Le full fine-tuning : l'approche naïve

**Idée** : ré-entraîner **tous les paramètres** du modèle sur ton dataset.

**Problème** : les LLM modernes ont **des milliards** de paramètres :

| Modèle | Paramètres | RAM GPU nécessaire (FP32) |
|---|---:|:---:|
| BERT-base | 110M | ~2 GB |
| GPT-2 | 1.5B | ~30 GB |
| Llama 3.2 1B | 1B | ~20 GB |
| Llama 3.3 70B | 70B | **~1.4 TB** 😱 |

Fine-tuner un Llama 70B complet = **multiple H100** (~50k€ matériel) sur plusieurs jours. **Inaccessible** pour 99% des équipes.

## 🪶 LoRA : la révolution 2021

**LoRA** (Low-Rank Adaptation) change tout. Au lieu de modifier **tous** les paramètres, on **gèle le modèle** et on ajoute de **petites matrices** entraînables.

### 🎨 L'intuition

Imagine une matrice de poids $W$ de taille $d \times d$ (disons 4096×4096 = 16M paramètres).

Au lieu de modifier $W$, on apprend une **modification** $\Delta W = B \cdot A$ où :
- $A$ a la taille $r \times d$ (petite)
- $B$ a la taille $d \times r$ (petite)
- $r$ (rang) est **petit** : typiquement 8, 16, 32, 64

Les poids effectifs deviennent : $W_{\text{adapted}} = W + B \cdot A$

In [ ]:
#| fig-cap: "LoRA : décomposition en matrices de rang faible"

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Matrice originale (figée)
axes[0].imshow(np.random.randn(20, 20), cmap="RdBu", aspect="auto")
axes[0].set_title("W (4096×4096)\n16 777 216 params — GELÉ", fontsize=11)
axes[0].set_xlabel("Paramètres frozen")

# Matrice A (entraînée)
axes[1].imshow(np.random.randn(4, 20), cmap="Greens", aspect="auto")
axes[1].set_title("A (r=8 × 4096)\n32 768 params — entraîné", fontsize=11, color="green")

# Matrice B
axes[2].imshow(np.random.randn(20, 4), cmap="Oranges", aspect="auto")
axes[2].set_title("B (4096 × r=8)\n32 768 params — entraîné", fontsize=11, color="darkorange")

plt.suptitle("LoRA : seulement ~65k params à entraîner au lieu de 16.7M (0.4%)",
              fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 💡 Pourquoi ça marche ?

**Hypothèse clé** : l'adaptation à une nouvelle tâche vit dans un **sous-espace de faible dimension**. On n'a pas besoin de modifier toutes les directions de $W$, seulement quelques-unes.

**Résultats empiriques** (papier LoRA 2021) :
- Qualité **quasi identique** au full fine-tuning
- **~0.1% des paramètres** entraînés
- **3× moins** de RAM
- Plusieurs **adapters** LoRA peuvent cohabiter (un par tâche)

## 📊 Comparaison des techniques

In [ ]:
#| fig-cap: "Comparaison : full fine-tuning vs LoRA vs prompt engineering"

techniques = ["Prompt\nengineering", "RAG", "LoRA", "Full\nfine-tuning"]
cout = [1, 5, 50, 5000]  # échelle arbitraire
temps = [0.01, 1, 10, 500]  # heures
qualite_spec = [40, 60, 85, 90]  # score arbitraire

fig, ax1 = plt.subplots(figsize=(11, 6))

x = np.arange(len(techniques))
w = 0.25

ax1.bar(x - w, cout, w, label="Coût (€, log)", color="coral")
ax1.set_yscale("log")
ax1.set_ylabel("Coût (€, échelle log)", color="coral")
ax1.set_xticks(x)
ax1.set_xticklabels(techniques)
ax1.tick_params(axis="y", labelcolor="coral")

ax2 = ax1.twinx()
ax2.bar(x, temps, w, label="Temps (h, log)", color="steelblue")
ax2.set_yscale("log")
ax2.set_ylabel("Temps (h, échelle log)", color="steelblue")
ax2.tick_params(axis="y", labelcolor="steelblue")

ax3 = ax1.twinx()
ax3.spines["right"].set_position(("axes", 1.1))
ax3.bar(x + w, qualite_spec, w, label="Qualité spécialisation", color="seagreen")
ax3.set_ylabel("Qualité (score arbitraire)", color="seagreen")
ax3.tick_params(axis="y", labelcolor="seagreen")

ax1.set_title("Coût vs temps vs qualité selon la technique", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**Message** : LoRA est le **sweet spot** quand on a vraiment besoin de fine-tuning — **qualité** quasi maximale, pour une **fraction** du coût.

# 3. PEFT : la librairie HuggingFace

**PEFT** (Parameter-Efficient Fine-Tuning) est la librairie officielle de HuggingFace pour LoRA et ses variantes.

## 🧪 Exemple minimal : LoRA sur un modèle

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM

# 1. Charger le modèle de base
modele = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",  # petit modèle pour la démo
    torch_dtype="auto",
    device_map="auto",
)

# 2. Configuration LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # type de tâche
    r=8,                            # rang
    lora_alpha=16,                  # scaling factor
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],  # quelles couches LoRA-ifier
)

# 3. Wrapper le modèle
modele_lora = get_peft_model(modele, lora_config)

# 4. Voir le gain
modele_lora.print_trainable_parameters()
# trainable params: 851,968 || all params: 1,236,662,272 || trainable%: 0.069

**Observation** : seulement **0.069%** des paramètres sont entraînables. Le reste est **gelé**.

## 🎛️ Les hyperparamètres LoRA clés

| Paramètre | Rôle | Valeurs typiques |
|---|---|---|
| **`r`** | Rang des matrices A et B | 8, 16, 32, 64 |
| **`lora_alpha`** | Scaling (souvent `= 2*r`) | 16, 32, 64 |
| **`lora_dropout`** | Régularisation | 0.05 - 0.1 |
| **`target_modules`** | Quelles couches adapter | `["q_proj", "v_proj"]` classique |

**Règle empirique** :
- `r` petit (8) → légère adaptation, peu de mémoire
- `r` grand (64) → plus de capacité, plus de mémoire
- Commencer à `r=8`, augmenter si underfit

# 4. Préparer un dataset de fine-tuning

## 📋 Le format JSONL

Le format standard est **JSONL** (un JSON par ligne) :

```jsonl
{"prompt": "Explique la photosynthèse", "completion": "La photosynthèse est le processus..."}
{"prompt": "Qu'est-ce que Python ?", "completion": "Python est un langage..."}
```

Pour un **chatbot**, on utilise plutôt le format **messages** :

```jsonl
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

## 🏗️ Créer un dataset synthétique TechVolt

On va simuler un dataset pour **fine-tuner** un petit modèle qui répond aux questions SAV de TechVolt dans un **format très précis**.

In [ ]:
# Dataset de fine-tuning TechVolt (synthétique)
dataset_techvolt = [
    {
        "instruction": "Le client signale que sa borne affiche E01. Réponds en format ticket.",
        "reponse": "TICKET-TECH\n-----------\nCode : E01\nDiagnostic : Surchauffe\nAction : Laisser refroidir 30 min\nUrgence : Normale"
    },
    {
        "instruction": "Le client signale que sa borne affiche E04. Réponds en format ticket.",
        "reponse": "TICKET-TECH\n-----------\nCode : E04\nDiagnostic : Court-circuit détecté\nAction : NE PAS UTILISER, contacter SAV immédiatement\nUrgence : CRITIQUE"
    },
    {
        "instruction": "Le client signale que sa borne affiche E02. Réponds en format ticket.",
        "reponse": "TICKET-TECH\n-----------\nCode : E02\nDiagnostic : Défaut de terre\nAction : Contacter un électricien certifié\nUrgence : Haute"
    },
    # ... imaginer 50+ exemples pour un vrai fine-tuning
]

# Sauvegarde en JSONL
chemin_dataset = TMP / "dataset_techvolt.jsonl"
with open(chemin_dataset, "w", encoding="utf-8") as f:
    for ex in dataset_techvolt:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# Relire pour vérifier
with open(chemin_dataset, encoding="utf-8") as f:
    for ligne in f:
        ex = json.loads(ligne)
        print(f"Instruction : {ex['instruction'][:60]}")
        print(f"Réponse     : {ex['reponse'][:60]}...\n")

## 🔢 Taille typique d'un dataset

**Règle empirique** :

| Taille dataset | Usage |
|---|---|
| **< 50 exemples** | Few-shot prompting, pas de fine-tuning |
| **50-500** | LoRA léger (format, style simple) |
| **500-5 000** | LoRA solide |
| **5 000-50 000** | LoRA avancé, résultats production |
| **> 50 000** | Full fine-tuning justifié |

## 🎯 Train/validation split

Comme toujours en ML, on sépare :

In [ ]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(dataset_techvolt, test_size=0.2, random_state=42)

print(f"Train : {len(train)} exemples")
print(f"Val   : {len(val)} exemples")

**Attention** : pour un dataset de 3 exemples c'est ridicule, mais c'est la méthodologie.

## 🏭 Avec datasets (HuggingFace)

In [ ]:
from datasets import Dataset

# Charger depuis JSONL
ds = Dataset.from_json(str(TMP / "dataset_techvolt.jsonl"))

# Split
ds = ds.train_test_split(test_size=0.2, seed=42)
print(ds)
# DatasetDict({
#     train: Dataset({features: ['instruction', 'reponse'], num_rows: 2}),
#     test: Dataset({features: ['instruction', 'reponse'], num_rows: 1})
# })

## ✏️ Exercice 1 — Créer un dataset de classification

> **ℹ️ Note**
>
## 📝 À faire

Tu veux fine-tuner un petit modèle pour classifier des avis TechVolt en 3 catégories : `POSITIF`, `NEUTRE`, `NEGATIF`.

1. Crée un dataset de **15 exemples** au format :
   ```json
   {"avis": "...", "classe": "POSITIF"}
   ```
   
2. Assure-toi d'avoir les **3 classes équilibrées** (5 de chaque)

3. Sauvegarde en JSONL

4. Charge avec HuggingFace `datasets`

5. Fais un split train/val 80/20

6. Affiche les **stats** (nb par classe dans chaque split)

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

# 5. Le fine-tuning en pratique

> **🎯 Important**
>
## ⚠️ Section à exécuter sur GPU
Les cellules qui suivent ne peuvent tourner que sur **GPU**. Utilise **Google Colab T4 (gratuit)** ou un PC avec CUDA.

Si tu n'as pas de GPU, lis pour **comprendre** — tu pourras exécuter plus tard.


## 🚀 Exemple complet (code type)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

# 1. Charger tokenizer + modèle
model_name = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

modele = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

# 2. Préparer dataset
def format_prompt(example):
    """Formate un exemple pour le training."""
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Réponse:\n{example['reponse']}"
    }

ds = Dataset.from_json(str(TMP / "dataset_techvolt.jsonl"))
ds = ds.map(format_prompt)

# 3. Tokeniser
def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=512, padding="max_length")

ds_tokenized = ds.map(tokenize, batched=True)

# 4. LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)
modele_lora = get_peft_model(modele, lora_config)
modele_lora.print_trainable_parameters()

# 5. Training
training_args = TrainingArguments(
    output_dir="./techvolt_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    warmup_steps=10,
    logging_steps=5,
    save_strategy="epoch",
    fp16=True,
)

trainer = Trainer(
    model=modele_lora,
    args=training_args,
    train_dataset=ds_tokenized,
    tokenizer=tokenizer,
)

trainer.train()

# 6. Sauvegarder l'adapter LoRA
modele_lora.save_pretrained("./techvolt_adapter")

**Résultat** : un dossier `./techvolt_adapter` contenant **uniquement** les poids LoRA (quelques MB, contre plusieurs GB pour le modèle complet).

## 🔌 Charger l'adapter pour inférence

In [ ]:
from peft import PeftModel

# Modèle de base
modele_base = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

# Appliquer l'adapter LoRA
modele_final = PeftModel.from_pretrained(modele_base, "./techvolt_adapter")

# Inférence
inputs = tokenizer(
    "### Instruction:\nLe client signale E03. Réponds en format ticket.\n\n### Réponse:\n",
    return_tensors="pt"
)
outputs = modele_final.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

**Sortie attendue** (après fine-tuning) :
```
TICKET-TECH
-----------
Code : E03
Diagnostic : Problème wifi
Action : Redémarrer la borne, voir fiche connectivité
Urgence : Normale
```

Le format est **exactement** celui appris. Un modèle sans fine-tuning **n'arriverait pas** à ce format strict sans beaucoup de prompting.

## 💡 Hyperparamètres clés

| Paramètre | Rôle | Valeurs types |
|---|---|---|
| **`num_train_epochs`** | Nb de passes sur le dataset | 3-10 |
| **`per_device_train_batch_size`** | Batch size | 1-8 (selon GPU) |
| **`learning_rate`** | Taux d'apprentissage | 1e-4 à 5e-4 pour LoRA |
| **`warmup_steps`** | Échauffement LR | 5-10% des steps |
| **`fp16`/`bf16`** | Mixed precision | Toujours activer |

# 6. Évaluer un modèle fine-tuné

## 🧪 Comment savoir si ton fine-tuning marche ?

**3 niveaux de validation** :

### 1. Validation de format (le basique)

Le modèle respecte-t-il le format attendu ? (JSON valide, structure ticket, etc.)

In [ ]:
def valider_format_ticket(sortie: str) -> bool:
    """Vérifie que le format TICKET-TECH est respecté."""
    required_fields = ["TICKET-TECH", "Code :", "Diagnostic :", "Action :", "Urgence :"]
    return all(f in sortie for f in required_fields)

# Évaluer sur un set de test
test_cases = [...]
resultats = []
for cas in test_cases:
    sortie = inference(cas["instruction"])
    resultats.append(valider_format_ticket(sortie))

print(f"Taux de format valide : {np.mean(resultats)*100:.1f}%")

### 2. Validation de contenu (le critique)

Le modèle donne-t-il la **bonne** info ?

In [ ]:
# Si tu as un dataset de test avec réponses attendues
def accuracy_champ(preds: list, vrais: list, champ: str) -> float:
    """Calcule l'accuracy sur un champ spécifique extrait des sorties."""
    import re
    corrects = 0
    for pred, vrai in zip(preds, vrais):
        match_pred = re.search(f"{champ} : (.+)", pred)
        match_vrai = re.search(f"{champ} : (.+)", vrai)
        if match_pred and match_vrai:
            if match_pred.group(1).strip() == match_vrai.group(1).strip():
                corrects += 1
    return corrects / len(preds)

### 3. Comparaison avec baseline (le commercial)

Ton modèle fine-tuné **bat-il** :
- Le modèle **de base** (sans fine-tuning) ?
- Un **gros modèle** avec bon prompt (ex: Llama 70B via Groq) ?

Si le modèle fine-tuné **ne bat pas** un gros modèle avec bon prompt → tu as **perdu ton temps**.

## 📊 Le piège classique : l'overfitting

Avec peu de données, le modèle peut **mémoriser** le train set sans généraliser.

**Signes d'overfitting** :
- Loss train très basse, loss val qui stagne
- Marche parfaitement sur les exemples d'entraînement
- Échoue lamentablement sur des exemples **légèrement différents**

**Solutions** :
- **Plus de données** (la seule vraie solution)
- **Early stopping** basé sur la val loss
- **`lora_dropout`** plus élevé (0.1-0.2)
- **Learning rate** plus petit

# 7. Alternatives modernes au fine-tuning classique

## 🎓 RLHF (Reinforcement Learning from Human Feedback)

C'est ce qui a transformé GPT-3 en **ChatGPT** :

1. **Supervised Fine-Tuning (SFT)** : fine-tuning classique sur exemples
2. **Reward Model** : un modèle apprend à **noter** les réponses (à partir de comparaisons humaines)
3. **Reinforcement Learning** : le LLM apprend à **maximiser** le score du reward model

**Résultat** : un modèle **aligné** sur les préférences humaines (utile, honnête, inoffensif).

**Inconvénient** : **complexe** et nécessite beaucoup de feedback humain.

## 🚀 DPO (Direct Preference Optimization) — 2023

**Alternative simplifiée** à RLHF : on peut optimiser les préférences **sans** passer par un reward model séparé.

Dataset : paires `(réponse_préférée, réponse_rejetée)`.

```json
{
    "prompt": "Explique les trous noirs à un enfant de 8 ans",
    "chosen": "Imagine une énorme aspirateur dans l'espace...",
    "rejected": "Les trous noirs sont des singularités gravitationnelles..."
}
```

**Beaucoup plus simple** à implémenter, **résultats similaires**.

**Libs** : `trl` (HuggingFace) implémente DPO en quelques lignes.

## 🤝 Constitutional AI (Anthropic, 2022)

Approche d'Anthropic pour entraîner Claude :

1. On donne au modèle une **"constitution"** (principes éthiques)
2. Le modèle **critique** ses propres réponses selon cette constitution
3. On **réentraîne** sur les réponses améliorées

**Particularité** : moins de besoin de feedback humain (le modèle s'auto-critique).

# 🏁 Exercice bilan — Décider de la stratégie

> **ℹ️ Note**
>
## 📝 Énoncé

Pour **chacun** des 5 cas suivants, décide :
- Prompt engineering
- RAG
- LoRA fine-tuning
- Full fine-tuning
- Autre chose ?

Et **justifie** en 2-3 phrases.

**Cas 1** : Tu veux que ton chatbot adopte le ton "bienveillant mais direct" de ta marque sur 20 exemples de dialogues.

**Cas 2** : Tu veux un chatbot qui réponde sur les 10 000 pages de documentation technique de ton produit.

**Cas 3** : Tu as 50 000 exemples de tickets support classés par catégorie (Bug / Feature / Question). Tu veux un classifier qui fait 5000 requêtes/jour à coût minimum.

**Cas 4** : Tu veux un assistant qui convertit du langage naturel en requêtes SQL sur **ton schéma de base** (20 tables métier spécifiques).

**Cas 5** : Tu veux que ton LLM parle comme Shakespeare.

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

# 🎓 Récapitulatif

## 📋 Ce que tu dois retenir

| Concept | Résumé |
|---|---|
| **Quand fine-tuner** | Format strict, domaine très spécialisé, coût/perf, comportement systématique |
| **Quand NE PAS fine-tuner** | Pour injecter des connaissances (→ RAG), ajuster un ton (→ prompt) |
| **LoRA** | Ajoute petites matrices A, B rank-r au lieu de modifier W |
| **`r`, `lora_alpha`** | r=8 défaut, alpha=2r |
| **PEFT** | Lib HuggingFace pour LoRA et variantes |
| **Dataset JSONL** | Format standard, 500-5000 exemples typique |
| **RLHF/DPO** | Apprendre les préférences humaines |

## 🧠 Les 5 réflexes à prendre

1. **Toujours prompt engineering d'abord** (gratuit, immédiat)
2. **RAG pour les connaissances** privées (pas fine-tuning)
3. **LoRA > full fine-tuning** dans 95% des cas
4. **Dataset qualité > quantité** (500 bons exemples > 5000 bruités)
5. **Baseline** d'abord : fine-tuning doit **battre** un gros LLM bien prompté

## 🚨 Les pièges à éviter

1. **Fine-tuning pour injecter des connaissances** → utilise RAG
2. **Dataset trop petit** (< 100) → overfitting garanti
3. **Pas de validation** → tu ne sais pas si ça marche
4. **`r` trop grand** → lent, pas toujours mieux que r=8
5. **Ignorer le coût d'inférence** → modèle fine-tuné peut être cher à servir

## 🎉 Module 7 terminé !

Tu maîtrises maintenant :
- ✅ Les **embeddings** (7.1)
- ✅ L'architecture **Transformer** (7.2)
- ✅ L'**API LLM** et le prompt engineering (7.3)
- ✅ Le **RAG** (7.4)
- ✅ Les **agents** et tool-use (7.5)
- ✅ Le **fine-tuning** et LoRA (7.6)

Tu as la **boîte à outils complète** d'un·e ingénieur·e IA générative.

## 🚀 La suite : le TP sommatif

Dans le [**TP sommatif Module 7**](tp_sommatif.qmd), tu vas construire un **chatbot RAG + agent complet** pour TechVolt, combinant **TOUT** ce qu'on a vu :
- Indexation d'une doc avec ChromaDB
- Retrieval + prompt RAG strict
- Agent qui enchaîne RAG + outils métier
- Évaluation et logging
- Interface en ligne de commande

C'est le **projet vitrine** à mettre sur ton CV. 💼

> **💡 Astuce**
>
## 💡 Défi pour consolider

**Cherche un modèle fine-tuné** sur [HuggingFace Hub](https://huggingface.co/models?sort=trending) qui correspond à un domaine qui t'intéresse (médical, juridique, code, écriture...).

Regarde :
- Quel modèle de base a été utilisé ?
- Quel dataset (taille, qualité) ?
- Comment est-il évalué ?
- Est-ce vraiment meilleur qu'un gros LLM généraliste bien prompté ?

**Astuce** : les modèles **les plus populaires** sur HF Hub sont souvent des LoRA (petit fichier d'adapter).